# Solution — Multi-API Kafka Pipeline: Consumers

Two consumers in this notebook:
- **Weather Consumer** — reads from `weather` topic, prints formatted conditions per city
- **Crypto Consumer** — reads from `crypto` topic, tracks price changes and fires alerts

> Run both consumers **before** starting the producers.

---
## Consumer 1 — Weather

In [ ]:
from kafka import KafkaConsumer
import json

BROKERS       = ['course-kafka:9092']
WEATHER_TOPIC = 'weather'

In [ ]:
# group_id='weather-monitor' — offset is committed automatically, so on restart
# the consumer picks up exactly where it left off instead of re-reading from the beginning.
weather_consumer = KafkaConsumer(
    WEATHER_TOPIC,
    group_id                = 'weather-monitor',
    bootstrap_servers       = BROKERS,
    auto_commit_interval_ms = 1000
)

In [ ]:
for message in weather_consumer:

    # Deserialize: bytes → str → dict
    d = json.loads(message.value.decode('utf-8'))

    # f-string with fixed-width fields keeps the output aligned even when
    # city names have different lengths.
    print(
        f"📍 {d['city']:<12} | "
        f"🌡  {d['temperature']:>5.1f}°C | "
        f"💨 {d['windspeed']:>5.1f} km/h | "
        f"{d['condition']}"
    )

---
## Consumer 2 — Crypto (with price change alerts)

In [ ]:
from kafka import KafkaConsumer
import json

BROKERS       = ['course-kafka:9092']
CRYPTO_TOPIC  = 'crypto'

# Fire an alert when price moves more than this percentage since the last message.
ALERT_CHANGE_PCT = 2.0

In [ ]:
crypto_consumer = KafkaConsumer(
    CRYPTO_TOPIC,
    group_id                = 'crypto-monitor',
    bootstrap_servers       = BROKERS,
    auto_commit_interval_ms = 1000
)

In [ ]:
# prev_prices stores the last seen price for each coin.
# Initialised as empty — the first message for each coin has no previous price to compare.
prev_prices = {}

for message in crypto_consumer:

    # Deserialize: bytes → str → dict
    d     = json.loads(message.value.decode('utf-8'))
    coin  = d['coin']
    price = d['price_usd']

    if coin not in prev_prices:
        # First time we see this coin — no previous price to compare against.
        # Store the price and skip the change calculation for this message.
        prev_prices[coin] = price
        print(f"🆕 {coin:<10} | ${price:>12,.2f} | (first message — no previous price)")
        continue

    # Calculate percentage change relative to the previous price.
    prev       = prev_prices[coin]
    change_pct = ((price - prev) / prev) * 100
    arrow      = '📈' if change_pct >= 0 else '📉'

    print(
        f"{arrow} {coin:<10} | "
        f"${price:>12,.2f} | "
        f"change: {change_pct:+.2f}%"
    )

    # Fire an alert if the absolute change exceeds the threshold.
    if abs(change_pct) >= ALERT_CHANGE_PCT:
        print(f"     ⚠️  PRICE ALERT: {coin} moved {change_pct:+.2f}% since last message!")

    # Always update the previous price AFTER printing,
    # so the next message compares against the most recent value.
    prev_prices[coin] = price